# 🧠 Transfer Learning: Backbone Freezing and Parameter Analysis

Welcome to the hands-on explanation notebook for **Transfer Learning**! In this notebook, we will:
1. Explain the theory of feature reuse and the hierarchy of layer transferability.
2. Load a standard neural network (`ResNet18`) representing a pre-trained model.
3. Write a parameter counting utility to measure model complexity.
4. Implement **backbone weight freezing** by setting gradient tracking parameters to `False`.
5. Modify the model's output head to suit a new custom classification task.
6. Contrast the number of trainable parameters before and after transfer learning freezing.
7. Connect this process to YOLO's default pre-trained initialization and its `freeze` training argument.

Let's start by importing the necessary libraries.

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

# Set seed for reproducibility
torch.manual_seed(42)

## 1. Implementing the Parameter Counter Utility

We write a helper function to calculate:
-   **Total Parameters:** The total number of weights and biases in the model.
-   **Trainable Parameters:** The parameters whose gradients are computed (where `requires_grad=True`), meaning they will be updated by the optimizer.

In [ ]:
def count_parameters(model):
    """
    Count total and trainable parameters in a PyTorch model.
    """
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

## 2. Loading ResNet18 Backbone

We instantiate a standard `ResNet18` model. By default, all layers are trainable.

In [ ]:
model = models.resnet18()

total_p, trainable_p = count_parameters(model)
print("--- Initial ResNet18 model ---")
print(f"Total Parameters    : {total_p:,}")
print(f"Trainable Parameters: {trainable_p:,}")

## 3. Implementing Transfer Learning (Freezing & Head Replacement)

Now we reuse the model for a target task containing **10 custom classes**:
1.  **Freeze Backbone:** Loop over all parameters and turn off gradient calculation (`requires_grad = False`). This freezes the feature extractor.
2.  **Replace Output Head:** Replace the final fully connected layer (`model.fc`) with a new linear layer. This new layer will automatically have `requires_grad = True` by default.

In [ ]:
# Step 1: Freeze all parameters
for param in model.parameters():
    param.requires_grad = False

total_p, trainable_p = count_parameters(model)
print("--- After Freezing Feature Extractor ---")
print(f"Total Parameters    : {total_p:,}")
print(f"Trainable Parameters: {trainable_p:,}")

# Step 2: Replace final fully connected layer
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 10)

total_p, trainable_p = count_parameters(model)
print("\n--- After Replacing Output Head (10 Classes) ---")
print(f"Total Parameters    : {total_p:,}")
print(f"Trainable Parameters: {trainable_p:,}")

Observe:
-   **Total Parameters:** Stays virtually identical ($\approx 11.2$ Million parameters).
-   **Trainable Parameters:** Dropped from $11,176,512$ down to only **$5,130$** (representing $512 \times 10$ weights $+ 10$ biases in the final layer).
-   **The Benefit:** Training is now extremely fast and requires much less memory because gradients are only calculated for the final output layer, keeping pre-trained feature extractors stable.

## 💡 Connection to YOLO and Deep Learning
*   **Default Pre-training:** When you run `yolo train model=yolo11n.pt`, YOLO automatically performs transfer learning, loading pre-trained weights from the COCO dataset to initialize the network.
*   **The `freeze` Argument:** If you want to freeze the backbone layers in YOLO to prevent them from changing, you can configure the `freeze` parameter:
    ```python
    from ultralytics import YOLO
    model = YOLO('yolo11n.pt')
    model.train(data='data.yaml', epochs=50, freeze=10) # Freezes the first 10 layers
    ```
    This is highly useful when fine-tuning on very small datasets to prevent overfitting.